In [1]:
from ax.service.ax_client import AxClient
import sys
sys.path.append('../')
import helper_functions as hf


In [2]:
iteration_to_update = 8




In [3]:
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,drug_conc,surf_1,surf_2
0,0,0_0,COMPLETED,GenerationStep_0,35.0,0.2063,0.3073,0.0373,29,6,25.0,s7,s6
1,1,1_0,COMPLETED,GenerationStep_0,50.0,0.4045,0.4196,0.0728,6,44,25.0,s4,s4
2,2,2_0,COMPLETED,GenerationStep_0,100.0,0.2962,0.4364,0.0493,4,27,25.0,s2,s6
3,3,3_0,COMPLETED,GenerationStep_0,92.0,0.3528,0.2810,0.0711,91,1,25.0,s1,s2
4,4,4_0,COMPLETED,GenerationStep_0,46.0,0.2063,0.3073,0.0373,32,14,25.0,s3,s4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,91,91_0,COMPLETED,GenerationStep_1,66.0,0.3528,0.2810,0.0711,8,58,25.0,s4,s8
92,92,88_0,COMPLETED,GenerationStep_1,23.0,0.2063,0.3073,0.0373,23,0,25.0,s7,s4
93,93,93_0,COMPLETED,GenerationStep_1,100.0,0.4045,0.4196,0.0728,5,16,25.0,s4,s8
94,94,90_0,COMPLETED,GenerationStep_1,31.0,0.2962,0.4364,0.0493,31,0,25.0,s7,s4


In [4]:
def update_data_to_optimizer(ax_client, list_of_new_failures, list_of_new_success):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    if len(list_of_new_failures) != 0:
        for trial_index in list_of_new_failures:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue

            # Build the forced-failure payload
            new_failure = {
                "obj_surf_conc": hf.surfactant_stock_conc,
            }
            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_failure)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {hf.surfactant_stock_conc}")

    if len(list_of_new_success) != 0:
        for trial_index in list_of_new_success:
            # Make sure we actually have this trial
            if trial_index not in trials_df["trial_index"].values:
                print(f"Trial {trial_index} not found – skipping.")
                continue    

            surf_conc_success = trials_df[trials_df['trial_index'] == trial_index]['surf_1_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_2_conc'].values[0] + trials_df[trials_df['trial_index'] == trial_index]['surf_3_conc'].values[0]
            new_success = {
                "obj_surf_conc": surf_conc_success,
            }

            # Update the trial in-place
            ax_client.update_trial_data(trial_index=trial_index, raw_data=new_success)
            print(f"Updated trial {trial_index}: set obj_surf_conc = {surf_conc_success}")

    ax_client.save_to_json_file(updated_path)
    return ax_client

In [5]:
list_of_new_failures = [77,81]


list_of_new_success = []

In [6]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from SUCCESS to FAILURE")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from SUCCESS to FAILURE


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,drug_conc,surf_1,surf_2
77,77,77_0,COMPLETED,GenerationStep_1,71.0,0.4045,0.4196,0.0728,9,62,25.0,s6,s1
81,81,81_0,COMPLETED,GenerationStep_1,68.0,0.4045,0.4196,0.0728,5,63,25.0,s6,s1


In [7]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)
print("Changing them from FAILURE to SUCCESS")

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_success)]
df

Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************
Changing them from FAILURE to SUCCESS


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,drug_conc,surf_1,surf_2


In [8]:
new_ax_client = update_data_to_optimizer(ax_client = ax_to_update, list_of_new_failures = list_of_new_failures, list_of_new_success = list_of_new_success)


[INFO 07-31 11:16:14] ax.service.ax_client: Added data: {'obj_surf_conc': (100.0, None)} to trial 77.
[INFO 07-31 11:16:14] ax.service.ax_client: Added data: {'obj_surf_conc': (100.0, None)} to trial 81.


Updated trial 77: set obj_surf_conc = 100
Updated trial 81: set obj_surf_conc = 100


In [9]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials[updated_trials["trial_index"].isin(list_of_new_failures + list_of_new_success)]

,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,drug_conc,surf_1,surf_2
77,77,77_0,COMPLETED,GenerationStep_1,100.0,0.4045,0.4196,0.0728,9,62,25.0,s6,s1
81,81,81_0,COMPLETED,GenerationStep_1,100.0,0.4045,0.4196,0.0728,5,63,25.0,s6,s1
